In [53]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load your preprocessed dataset
df = pd.read_csv('political_wiki_human_ai_dataset.csv')


In [55]:
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'],
    test_size=0.2,  # 20% for testing
    random_state=42,
    stratify=df['label']  # keeps class balance
)


In [57]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    ngram_range=(1,2),  # unigrams and bigrams
    min_df=3,           # only include words in at least 3 docs
    max_df=0.9,         # exclude very common words
    max_features=10000  # adjust if memory is an issue
)

# Fit on training, transform both train and test
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)


In [59]:
print(vectorizer.vocabulary_["nato"])

5508


In [61]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train_vec, y_train)


LogisticRegression(max_iter=1000, random_state=42)

In [63]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

y_pred = clf.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=['Human', 'AI']))


Accuracy: 0.9965034965034965

Confusion Matrix:
 [[284   2]
 [  0 286]]

Classification Report:
               precision    recall  f1-score   support

       Human       1.00      0.99      1.00       286
          AI       0.99      1.00      1.00       286

    accuracy                           1.00       572
   macro avg       1.00      1.00      1.00       572
weighted avg       1.00      1.00      1.00       572



In [65]:
import joblib
joblib.dump(clf, 'tfidf_logreg_model.joblib')
joblib.dump(vectorizer, 'tfidf_vectorizer.joblib')


['tfidf_vectorizer.joblib']

In [67]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=['Human', 'AI']))


Accuracy: 0.9965034965034965

Classification Report:
               precision    recall  f1-score   support

       Human       1.00      0.99      1.00       286
          AI       0.99      1.00      1.00       286

    accuracy                           1.00       572
   macro avg       1.00      1.00      1.00       572
weighted avg       1.00      1.00      1.00       572



In [69]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(clf, vectorizer.transform(df['text']), df['label'], cv=5)
print("Cross-validation accuracy:", scores.mean())


Cross-validation accuracy: 0.9951048951048952


In [71]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load your dataset
df = pd.read_csv('political_wiki_human_ai_dataset.csv')

# Split the data exactly as you did for training/testing
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)

# You need the test set's indices to get the titles!
test_indices = y_test.index

# Predict using your model (clf, vectorizer as before)
X_test_vec = vectorizer.transform(X_test)
y_pred = clf.predict(X_test_vec)

# Build a DataFrame for easy comparison
test_results = pd.DataFrame({
    'title': df.loc[test_indices, 'title'].values,
    'true_label': y_test.values,
    'pred_label': y_pred,
    'text': X_test.values
})

# Show misclassified samples
misclassified = test_results[test_results['true_label'] != test_results['pred_label']]

print("Misclassified samples (title and true/predicted labels):")
print(misclassified[['title', 'true_label', 'pred_label']])


Misclassified samples (title and true/predicted labels):
                          title  true_label  pred_label
38    Royal Jordanian Air Force           0           1
108  Foreign relations of Samoa           0           1


In [ ]:
# BERT Fine Tuning

In [ ]:
!pip install transformers torch scikit-learn


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

# Load your dataset
df = pd.read_csv('political_wiki_human_ai_dataset.csv')

# Split
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)

# Tokenization
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(texts):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

train_encodings = tokenize_function(X_train)
test_encodings = tokenize_function(X_test)

# Dataset class
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = TextDataset(train_encodings, y_train)
test_dataset = TextDataset(test_encodings, y_test)

# Model & training args
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

training_args = TrainingArguments(
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    evaluation_strategy="epoch",
    logging_steps=10,
    output_dir='./results',
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

# Evaluate
preds = trainer.predict(test_dataset)
import numpy as np
from sklearn.metrics import classification_report

y_pred = np.argmax(preds.predictions, axis=1)
print(classification_report(list(y_test), y_pred, target_names=['Human', 'AI']))


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [19]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('political_wiki_human_ai_dataset.csv')

X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)


In [21]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_texts(texts, max_length=256):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )

train_encodings = tokenize_texts(X_train)
test_encodings = tokenize_texts(X_test)


In [23]:
import torch

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = TextDataset(train_encodings, y_train)
test_dataset = TextDataset(test_encodings, y_test)


In [ ]:
!pip install --upgrade transformers


In [25]:
from transformers import BertForSequenceClassification, Trainer, TrainingArguments

model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

training_args = TrainingArguments(
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    evaluation_strategy="epoch",
    save_strategy="no",
    logging_steps=10,
    output_dir='./results',
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=None,
)
trainer.train()


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [17]:
import transformers
print(transformers.__version__)

4.53.0


In [ ]:
import transformers
print(transformers.__file__)

In [3]:
import sys
print(sys.executable)
import transformers
print(transformers.__version__)
print(transformers.__file__)


F:\Anaconda\python.exe
4.53.0
F:\Anaconda\Lib\site-packages\transformers\__init__.py


In [7]:
F:\Anaconda\python.exe -m pip uninstall transformers -y
F:\Anaconda\python.exe -m pip install transformers

SyntaxError: unexpected character after line continuation character (2411742459.py, line 1)

In [9]:
!pip uninstall transformers -y
!pip install transformers


Found existing installation: transformers 4.53.0
Uninstalling transformers-4.53.0:
  Successfully uninstalled transformers-4.53.0
  Using cached transformers-4.53.0-py3-none-any.whl.metadata (39 kB)
Using cached transformers-4.53.0-py3-none-any.whl (10.8 MB)


In [1]:
import transformers
print(transformers.__version__)


4.53.0


In [3]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    evaluation_strategy="epoch",
    logging_steps=10,
    output_dir='./results',
    report_to='none'
)
print(training_args)


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'